[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/astheeggeggs/testing_colab/blob/main/LDSC_Practical_1_Colab.ipynb)

# LD-score regression (LDSC) practical 1: SNP Heritability $\left(h_{\text{SNP}}^2\right)$

Before we get cracking on the practical, we need to get our environment set up, and all the required R packages installed.

Environment setup and installing the required packages

This Colab notebook uses an **R runtime**. We install the packages needed for the practical from CRAN and GitHub.

Packages used here:
- `data.table`
- `dplyr`
- `remotes`
- `GenomicSEM`


In [ ]:
# System dependencies that are often useful for R package installation
# (safe to run even if some packages are already present)
system("apt-get update -qq")
system("apt-get install -y -qq libcurl4-openssl-dev libxml2-dev libssl-dev")
system('python -m pip -q install gdown')
system('gdown --folder "https://drive.google.com/drive/folders/1BIBaoLlodLG_ni9DASh_bw-3RzUZSSJe?usp=sharing" -O /content/LDSC_Practical_1')
root_dir <- "/content/LDSC_Practical_1"
sumstat_dir <- file.path(root_dir, "EUR")
ref_data_dir <- file.path(root_dir, "EUR", "eur_w_ld_chr")

In [ ]:
# Install CRAN packages
install.packages(
  c("data.table", "dplyr", "remotes"),
  repos = "https://cloud.r-project.org",
  quiet=TRUE
)

# Install GenomicSEM from GitHub
# GenomicSEM is maintained on GitHub and its README recommends installation from GitHub.
remotes::install_github("GenomicSEM/GenomicSEM", upgrade = "never", quiet=TRUE)

suppressWarnings(suppressPackageStartupMessages({
  library(data.table)
  library(dplyr)
  library(remotes)
  library(GenomicSEM)
}))


In [ ]:
dir("/content/LDSC_Practical_1")
setwd("/content/LDSC_Practical_1")

### ✅ Load packages

If the install cell ran successfully, load the packages here!


In [ ]:
library(data.table)
library(dplyr)
library(GenomicSEM)

## Only **two** primary steps to run LDSC
We are using `GenomicSEM` library in R to run LDSC

1.   Munge the summary statistics: `munge()`.
*munge = convert raw data from one form to another*
2.   Run LD-score regression: `ldsc()`

## Required information in the summary statistics file
The summary statistics files input to `munge()` needs to contain a minimum of five pieces of information:


1.   The rsID of the SNP.
2.   An A1 allele column, indicating the effect allele.
3.   An A2 allele column, indicating the non-effect allele.
4.   A signed ($\pm$) effect column.
5.   The $P$-value associated with this effect.

## Arguments for the `munge()` function
The `munge()` function takes six arguments:
1. `files`: The name of the summary statistics files
2. `hm3`: The name of the reference file. Here we use Hapmap 3 SNPs.
3. `trait.names`: The trait names that will be used to name the saved files.
4. `N`: The sample sizes associated with the traits.
5. `info.filter`: INFO filter. Package default is to retain SNPs with `INFO > 0.9`.
6. `maf.filter`: MAF filter. Package default is to retain SNPs with `MAF > 0.01`.

## Arguments for the `ldsc()` function

The `ldsc()` function takes six arguments:

1.   `traits`: a vector of file names/paths to files which point to the munged sumstats.
2.   `sample.prev`: A vector of sample prevalences of length equal to the number of
traits. If the trait is continuous, the values should equal `NA`.
3.   `population.prev`: A vector of population prevalences. If the trait is continuous
the values should equal NA.
4.   `ld`: A folder of LD scores used as the independent variable in LDSC
5.   `wld`: A folder of LDSC weights (Typically the same folder as specified for the `ld` argument)
6.   `trait.names`: The trait names.

The code below assumes you have a folder with this structure:

```text
LDSC_Practical_1/
  EUR/
    w_hm3.snplist
    eur_w_ld_chr/
    CHR22only_Meta-analysis_Locke_et_al+UKBiobank_2018_UPDATED.txt.gz
    Meta-analysis_Locke_et_al+UKBiobank_2018_UPDATED.txt.gz
    Munged_BMI_Meta-analysis_Locke_et_al+UKBiobank_2018_for_LDSC.sumstats.gz
    BMI.sumstats.gz
```

Note that this should all be in place, as we've copied an entire folder across from google drive with the expected folder structure.


In [ ]:
# Set directories
sumstat_dir  <- "EUR/"
ref_data_dir <- "EUR/eur_w_ld_chr/"

The reference directory includes the HapMap3 (HM3) SNP IDs and their pre-computed LD scores.

In [ ]:
# Quick check
getwd()
dir(ref_data_dir)

HapMap3 SNPs and LD scores in European ancestry

For LDSC to work as expected, the included SNPs must be QCed. To ensure that, LDSC is often done using only the HM3 SNPs because
- HM3 SNPs are expected to be common variants (MAF > 0.01).
- HM3 SNPs typically have high imputation quality.

So, if the GWAS sum stats do not have information on these QC metrics, we may run LDSC using only the HM3 SNPs.

Also, pre-computed LD scores for these SNPs (in European ancestry) were provided with the original LDSC program - so that makes it much easier to run LDSC in European ancestry!

Let's inspect the HM3 SNP list that we'll pass to LDSC.


**Question:** How many SNPs are there in the HapMap3 reference file?

<details>
<summary>💡 Click here to reveal the answer!</summary>

There are 1,217,311 SNPs included in the HM3 reference file.

Note that there are 1,217,312 lines in `w_hm3.snplist` file, but this number of lines also includes the first line with column names. So, we subtract 1 to get the number of rows (= SNPs) in the dataset.

</details>


In [ ]:
hm3_file <- file.path(ref_data_dir, "w_hm3.snplist")

# Take a look at the first few lines
cat(system(paste("head", hm3_file), intern = TRUE), sep = "\n")

# Count the number of lines
cat(system(paste("wc -l", hm3_file), intern = TRUE), sep = "\n")

We will still restrict the LDSC analyses to the HM3 SNPs in this practical because we have the pre-computed LD scores for these SNPs only. Alternatively, one may compute LD scores for all QC-ed SNPs in the GWAS sample or an external LD reference panel.

## QC filters for LDSC

We may specify additional QC filters.

We will drop the SNPs with imputation $R^2 < 0.90$ or $MAF < 0.01$.

These filters will be applied only if the GWAS sum stats include information on these QC metrics. Otherwise, these filters are ignored.

In [ ]:
info_cutoff <- 0.9
maf_cutoff  <- 0.01

We will perform the analyses in two steps:

1. Munge the data: `munge()`

This step "cleans" your GWAS sum stats file to select (and rename if needed) the required columns.
This step also filters the SNPs to keep only the SNP IDs listed in the HM3 reference file.

This filtering is important because all SNPs retained in the munged data must have their LD scores computed and stored in the reference directory.

If the sum stats dataset includes QC metrics, this step will also apply the QC filters.

At the end of this step, the munged dataset will be saved in the specified location.

2. LD score regression: `ldsc()`

This is the main LDSC analysis using the munged dataset.

The results are saved as a log file in the specified location.

## BMI example: chromosome 22

For the tutorial, we practice `munge()` using only chromosome 22.

LDSC needs these column names in the munged data:

*   `SNP` - SNP identifier (e.g., rsID).
*    `N` - sample size (which may vary from SNP to SNP).
*   `BETA` - Beta. (or Z-score, or OR).**Signed with respect to A1** (Warning: Possible gotcha!)
* `A1` - first allele (effect allele).
* `A2` - second allele (other allele).
* `P-Value.`
* Any QC metrics, e.g., `INFO` or `MAF`.

Examine the BMI gwas sumstats (the chromosome 22 subset) prior to munging.

In [ ]:
bmi_file <- file.path(
  sumstat_dir,
  "CHR22only_Meta-analysis_Locke_et_al+UKBiobank_2018_UPDATED.txt.gz"
)
cat(system(paste0("zcat ", bmi_file, " | head"), intern=TRUE), sep="\n")

In this code, we want to verify which allele is the "effect allele" with respect to which the association statistic `BETA` is signed (positive or negative).

Look at the columns in the BMI sumstats.

Are the column names informative about the "effect allele"?

In this case, **YES**, as the effect allele is labeled `Tested_Allele`!

This is helpful, as we do not have a README file.

If the column names are uninformative and there is no README file, do not hesitate to verify with the corresponding author!

Using the following block of code, we rename the columns to what LDSC expects and save the file with the new column names.
Note that renaming the columns is not required if the sum stats file has standard column names that the program can interpret correctly - which is the case here. However, that may not always be the case.

In [ ]:
bmi_GWAS <- fread(bmi_file, header = TRUE, data.table = FALSE)
str(bmi_GWAS)
bmi_GWAS <- dplyr::rename(
  bmi_GWAS,
  A1 = Tested_Allele,
  A2 = Other_Allele
)
head(bmi_GWAS)

In [ ]:
bmi_updated_file <- file.path(sumstat_dir, "BMI_GWAS_for_LDSC.txt")
fwrite(
  bmi_GWAS,
  file = bmi_updated_file,
  sep = "\t",
  col.names = TRUE,
  row.names = TRUE,
  quote = FALSE
)
cat(system(paste0("head ", bmi_updated_file), intern=TRUE), sep='\n')


Some GWAS sumstats may report the "effect allele frequency" rather than MAF. In that case, we'd need to convert the effect allele frequency to MAF.

## Munge
Now, munge the BMI sum stats using the following code.

Provide the file path and name to save the munged summary statistics.

In [ ]:
library(GenomicSEM)
bmi_munged <- file.path(sumstat_dir, "munged_BMI_GWAS_chr22_for_LDSC")

munge(
  files = bmi_updated_file,
  hm3 = hm3_file,
  trait.names = bmi_munged,
  info.filter = info_cutoff,
  maf.filter = maf_cutoff
)

Examine the log file and answer the following questions

### ❓ Question
Which column is interpreted as the effect allele (A1)?

<details>
<summary>💡 Click here to reveal the answer!</summary>

The A1 column, which is the correct column.

Remember that we renamed the `Tested_Allele` column as `A1`

</details>

### ❓ Question
How many SNPs were present in the raw sum stats file?

<details>
<summary>💡 Click here to reveal the answer!</summary>

There were 29,991 SNPs present in the raw sum stats file.

</details>

### ❓ Question
How many SNPs were dropped because these are not present in the reference HapMap3 file?

<details>
<summary>💡 Click here to reveal the answer!</summary>

Answer: The A1 column, which is the correct column.

Remember that we renamed the "Tested_Allele" column as "A1"

</details>

### ❓ Question
How many SNPs were dropped due to low imputation quality (i.e., INFO)?

<details>
<summary>💡 Click here to reveal the answer!</summary>

Answer: None, as there was no INFO column in the raw dataset.

</details>

### ❓ Question
How many SNPs were dropped due to low or missing MAF?

<details>
<summary>💡 Click here to reveal the answer!</summary>

1 SNP was dropped due to low or missing MAF.

Usually, there will be few HapMap3 SNPs with MAF < 0.01 - which is why only one SNP was dropped here.

</details>

### ❓ Question
How many SNPs are left in the munged sum stats file?

<details>
<summary>💡 Click here to reveal the answer!</summary>

Answer: 14,382 SNPs are left in the munged sumstats file.
</details>


Now we are done munging the BMI sumstats (chr 22 subset).

The full BMI summary statistics are already munged for the practical (for all chromosomes) - how convenient! We'll compare the raw file with the munged file and then run LDSC.

In [ ]:
raw_bmi    <- file.path(sumstat_dir, "Meta-analysis_Locke_et_al+UKBiobank_2018_UPDATED.txt.gz")
munged_bmi <- file.path(sumstat_dir, "Munged_BMI_Meta-analysis_Locke_et_al+UKBiobank_2018_for_LDSC.sumstats.gz")

# cat(system(paste0("zcat ", raw_bmi, " | head"), intern=TRUE), sep='\n')
# cat(system(paste0("zcat ", munged_bmi, " | head"), intern=TRUE), sep='\n')

# cat(system(paste0("zcat ", raw_bmi, " | wc -l"), intern=TRUE), sep='\n')
# cat(system(paste0("zcat ", munged_bmi, " | wc -l"), intern=TRUE), sep='\n')

Look at the head (i.e., the first few lines) of the munged sum stats.

### ❓ Question
What are the columns included in the munged sumstats?

<details>
<summary>💡 Click here to reveal the answer!</summary>

`SNP`, `N`, `Z`, `A1`, `A2`
</details>

Now run the LDSC analysis of BMI using the munged genome-wide sumstats.

In [ ]:
bmi_ldsc_out <- file.path(sumstat_dir, "BMI")

ldsc(
  traits = munged_bmi,
  sample.prev = NA,
  population.prev = NA,
  ld = ref_data_dir,
  wld = ref_data_dir,
  trait.names = "BMI",
  ldsc.log = bmi_ldsc_out
)


**Note:** the `ldsc()` function in `GenomicSEM` is designed for bivariate LDSC, so it may print a warning about needing two or more traits. In this practical, that warning can be ignored.
This is okay. We will run multivariate LDSC in the next session.

The output log file also includes genetic correlations. These are not relevant to the current univariate analyses but will be covered in the next session.

Take a look at the log file (spat out in the output of the previous cell), and answer the following questions:

### ❓ Question
What are the Mean Chi^2 and the Lambda GC? Do these suggest an inflation of GWAS statistics?

<details>
<summary>💡 Click here to reveal the answer!</summary>

*   Mean $\chi^2$ = 3.9345.
*   $\lambda_{GC}$ = 2.7889.

Both are $>1$ and so suggest inflation of GWAS statistics.

However, this inflation may be due to true genetic associations (polygenicity and large enough sample size) and potential confounding (e.g., population stratification).
</details>

### ❓ Question
What is the estimate of the LDSC intercept in this analysis? Does it suggest significant bias(es) in the BMI GWAS results?

<details>
<summary>💡 Click here to reveal the answer!</summary>

LDSC intercept = 1.0202 (SE 0.0277).

As the LDSC intercept is not significantly greater than 1, it suggests that there was no significant confounding (population stratification) in the GWAS.
</details>

### ❓ Question
What is the estimated SNP heritability ($h^2_{\text{SNP}}$) of BMI?

<details>
<summary>💡 Click here to reveal the answer!</summary>

$h^2_{\text{SNP}}$ of BMI = 0.2091 (SE 0.0063)
</details>

Congratulations! You successfully ran the LDSC analyses of BMI.


Please note that LDSC uses the provided [munged] sum stats to estimate the heritability/variance per SNP. The program then uses the estimate of the per-SNP heritability to compute the total variance across all SNPs in the genome [i.e., the SNP heritability].

If the sum stats used to run LDSC analyses are NOT a random subset of all SNPs across the genome, the estimated per-SNP variance and, thus, the SNP-based heritability will be biased.

## Bonus (Time Permitting): LDSC with sumstats from Linear Mixed Model (LMM) GWAS

We will now compare the results of LDSC analyses using BMI GWAS sum stats from the Pan-UKBB project.

These analyses used SAIGE to perform linear mixed models GWAS and include related individuals.

The sum stats have already been munged. The sum stats did not include the sample size, so we have specified the $N$ (419,163) when munging.

In [ ]:
munged_lmm_bmi <- file.path(sumstat_dir, "BMI.sumstats.gz")

cat(system(paste0("zcat ", munged_lmm_bmi, " | head"), intern=TRUE), sep='\n')

bmi_lmm_ldsc_out <- file.path(sumstat_dir, "BMI_LMM")

ldsc(
  traits = munged_lmm_bmi,
  sample.prev = NA,
  population.prev = NA,
  ld = ref_data_dir,
  wld = ref_data_dir,
  trait.names = "BMI_lmm",
  ldsc.log = bmi_lmm_ldsc_out
)


### ❓ Question
What is the estimate of the LDSC intercept? Does it suggest significant bias(es) in the BMI GWAS results?

<details>
<summary>💡 Click here to reveal the answer!</summary>

 The LDSC intercept is 1.2284 (S.E. = 0.057) - which is significantly $>1$.

However, LDSC analyses of LMM sum stats may yield intercepts $>1$
For traits with high SNP heritability
In large samples (such as the UK Biobank, $N > 400k$)

See Loh, PR., Kichaev, G., Gazal, S. et al. Mixed-model association for biobank-scale datasets. Nat Genet 50, 906–908 (2018). https://doi.org/10.1038/s41588-018-0144-6 for further details.
</details>



In this case, we may want to consider some additional points:

Effective $N$ (sometimes referreed to as $N_{\text{eff}}$ is less than the raw $N$ due to relatedness between some participants.

Could there be residual confounding? Genetic related matrix used as a random effect in LMMs may not control for environmental confounding between family members.

However, note that the attenuation ratio is 0.097 (SE = 0.0242), which is totally acceptable.

## End

You have now run the full Colab version of Practical 1 for SNP heritability.
